## DC Steady Plasma Model (1D in z)

### Unknown fields
We solve for five steady-state fields on a 1D domain z in [0, L]:

$$
n_i(z),\; n_e(z),\; V(z),\; \Gamma_i(z),\; \Gamma_e(z).
$$

The electric field is defined as:

$$
E(z) \equiv -\frac{dV}{dz}.
$$

---

## Governing equations (as implemented)

(1) Ion continuity (steady)

$$
-\frac{d\Gamma_i}{dz} + \alpha(E)\,\Gamma_e = 0.
$$

(2) Electron continuity (steady)

$$
-\frac{d\Gamma_e}{dz} + \alpha(E)\,\Gamma_e = 0.
$$

(3) Poisson equation

$$
\frac{d^2 V}{dz^2} + \left(n_i - n_e\right) = 0.
$$

(4) Ion flux closure (drift–diffusion)

$$
\Gamma_i - \left(\mu_i(E)\,E\,n_i - D_i\,\frac{dn_i}{dz}\right) = 0.
$$

(5) Electron flux closure (drift–diffusion)

$$
\Gamma_e - \left(-E\,n_e - D_e\,\frac{dn_e}{dz}\right) = 0.
$$

---

## Coefficient models

Ion mobility mu_i(E)

$$
\mu_i(E)=\frac{0.5740}{1+0.66\sqrt{|E\,E_0|\cdot 10^{-2}}}\cdot \frac{1}{u_e}.
$$

Ionization coefficient alpha(E)

$$
\alpha(E)=\frac{2922\exp\!\left(-26.64\sqrt{\frac{1}{|E\,E_0/100|}}\right)}{\alpha_0}.
$$

Note: In the implementation, a small constant is added inside absolute values/denominators to avoid division-by-zero.

---

## Boundary conditions (Dirichlet-type, enforced via hard constraints)

$$
n_i(0)=n_{i,0},\quad n_i(L)=n_{i,L}
$$

$$
n_e(0)=n_{e,0},\quad n_e(L)=n_{e,L}
$$

$$
V(0)=V_{0},\quad V(L)=V_{L}
$$

$$
\Gamma_i(0)=\Gamma_{i,0},\quad \Gamma_i(L)=\Gamma_{i,L}
$$

$$
\Gamma_e(0)=\Gamma_{e,0},\quad \Gamma_e(L)=\Gamma_{e,L}
$$

---

## Nondimensionalization (DC code form)

Reference scales:

$$
t_0 = \frac{\varepsilon_0}{q_e\,n_0\,u_e},\qquad
R_0 = \frac{\varepsilon_0\,E_0}{q_e\,n_0},\qquad
\alpha_0=\frac{1}{R_0}.
$$

Normalization:

$$
z \leftarrow \frac{z}{R_0},\qquad
V \leftarrow \frac{V}{E_0R_0},\qquad
n \leftarrow \frac{n}{n_0},\qquad
\Gamma \leftarrow \frac{\Gamma}{u_e E_0 n_0},
$$

$$
D_{e,i} \leftarrow \frac{D_{e,i}}{u_e E_0 R_0}.
$$

In [1]:
import os
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID" 
# os.environ["CUDA_VISIBLE_DEVICES"]="3"#gpu_2_1
import re
string = os.popen('nvidia-smi -L').read()
gpus = re.findall('\(([^)]+)', string[string.find("GPU 2"):string.find("GPU 3")])
gpu_2 = gpus[0][6:]
gpu_2_1 = gpus[1][6:]
gpu_2_2 = gpus[2][6:]
os.environ["CUDA_VISIBLE_DEVICES"]=gpu_2_1
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="False"

In [2]:
#!/usr/bin/env python
# coding: utf-8
# ============================================================
# DC Steady PINN (ICP framework) - Full Notebook Script
# - PirateNet (Flax) + RWF + RFF + coord_norm
# - DC nondimensionalization
# - DC hard BC (linear correction in exp-space)
# - Autograd residuals (5 terms)
# - SAW (GradNorm) for 5 residuals (+ optional data)
# - Resampling + caching collocation
# - Save: ckpt / npz / plots / hash
# - Data loss: only (n_i, n_e, V) when N_lab > 0
# ============================================================

#%% ==========================================================
# 0) Experiment paths / configs
# ==========================================================
from pathlib import Path
import os, time, math, pickle, hashlib
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import scipy.constants

import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax import lax
import flax.linen as nn
from flax import struct, serialization
from flax.training import train_state as flax_train_state
from flax.training import checkpoints
from flax.traverse_util import flatten_dict

import optax
from soap_jax import soap
from tqdm.auto import trange

In [3]:
jax.config.update("jax_enable_x64", True)
DTYPE = jnp.float64
key = jax.random.PRNGKey(0)

In [4]:
# ---- user configs ----
experiment_folder = "./exp_dc_pirate_saw"   # <- 변경 가능
data_dir = Path("./data")                  # <- DC 데이터 경로
CKPT_DIR_NAME = "ckpts_dc_pirate_saw"
EPOCHS  = 300000 + 1

WIDTH   = 256
DEPTH   = 2
ACT     = "swish"

LR      = 1e-3
N_r     = 1000
N_lab   = 0                  # 0이면 interior data loss OFF, >0이면 (n_i,n_e,V) data loss ON
RESAMPLE_COLL_EVERY = 100

NPZ_SAVE_EVERY   = 1000
EXTRA_CKPT_EVERY = 1000000
PLOT_EPOCHS      = [10000, 50000, 100000, 150000, 200000, 250000, 300000]

In [5]:
# ---- dirs ----
exp = Path(experiment_folder).resolve()
exp.mkdir(parents=True, exist_ok=True)

CKPT_DIR = exp / CKPT_DIR_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = CKPT_DIR / "best_params.npz"

CKPT_SAVED_DIR = exp / "CKPT_SAVED"
CKPT_SAVED_DIR.mkdir(parents=True, exist_ok=True)

def _tag(step: int) -> str:
    return f"step{step:07d}"

def _now():
    return time.strftime("%Y%m%d-%H%M%S")

def epoch_dir(step: int) -> Path:
    d = CKPT_SAVED_DIR / f"step_{step:07d}"
    d.mkdir(parents=True, exist_ok=True)
    return d

In [6]:
#%% ==========================================================
# 1) DC nondimensionalization (keep DC exactly)
# ==========================================================
u_e = 20.0
D_e_SI = 50.0
D_i_SI = 0.01
gam = 0.01
V_dc_SI = -1000.0

eps0 = scipy.constants.epsilon_0
qe   = scipy.constants.elementary_charge

L_SI = 2e-2
T_SI = 1e-8

n0d = 1e16
E0d = 1e5

t0d = eps0/qe/n0d/u_e
R0d = eps0/qe*E0d/n0d
alpha0d = 1.0/R0d

D_e = D_e_SI/u_e/E0d/R0d
D_i = D_i_SI/u_e/E0d/R0d
V_dc = V_dc_SI/R0d/E0d
Lz = L_SI/R0d
T  = T_SI/t0d

phys_dc = dict(
    u_e=float(u_e), D_e=float(D_e), D_i=float(D_i), gam=float(gam),
    V_dc=float(V_dc), n0d=float(n0d), E0d=float(E0d), t0d=float(t0d),
    R0d=float(R0d), alpha0d=float(alpha0d), Lz=float(Lz), T=float(T),
)
print("[DC phys]", phys_dc)

[DC phys] {'u_e': 20.0, 'D_e': 0.04523782044931957, 'D_i': 9.047564089863914e-06, 'gam': 0.01, 'V_dc': -18.095128179727826, 'n0d': 1e+16, 'E0d': 100000.0, 't0d': 2.7631746790285554e-10, 'R0d': 0.000552634935805711, 'alpha0d': 1809.5128179727826, 'Lz': 36.19025635945565, 'T': 36.19025635945565}


In [7]:
#%% ==========================================================
# 2) Load DC data: grids.pkl + results_last_iters.pkl (steady)
# ==========================================================
with open(data_dir/"grids.pkl", "rb") as f:
    grids = pickle.load(f)
z_test_nV_SI = np.asarray(grids["nV"])     # SI grid for n/V
z_test_GE_SI = np.asarray(grids["GE"])     # SI grid for Gamma/E

with open(data_dir/"results_last_iters.pkl", "rb") as f:
    res = pickle.load(f)

t_ind = -1
gt = dict(
    z_nV_SI=z_test_nV_SI,
    z_GE_SI=z_test_GE_SI,
    n_i=np.asarray(res["n_i"][t_ind]),
    n_e=np.asarray(res["n_e"][t_ind]),
    V  =np.asarray(res["V"][t_ind]),
    Gamma_i=np.asarray(res["Gamma_i"][t_ind]),
    Gamma_e=np.asarray(res["Gamma_e"][t_ind]),
)

# nondim coords (training/inference)
z_test_nV = jnp.asarray(z_test_nV_SI / R0d, dtype=DTYPE)  # (NnV,)
z_test_GE = jnp.asarray(z_test_GE_SI / R0d, dtype=DTYPE)  # (NGE,)

# BC from ends (nondim)
def _bc_ends(arr):
    return float(arr[0]), float(arr[-1])

BC_VALUES = {
    "n_i_0": _bc_ends(gt["n_i"])[0] / n0d,
    "n_i_L": _bc_ends(gt["n_i"])[1] / n0d,
    "n_e_0": _bc_ends(gt["n_e"])[0] / n0d,
    "n_e_L": _bc_ends(gt["n_e"])[1] / n0d,
    "V_0":   _bc_ends(gt["V"])[0]   / (E0d*R0d),
    "V_L":   _bc_ends(gt["V"])[1]   / (E0d*R0d),
    "G_i_0": _bc_ends(gt["Gamma_i"])[0] / (u_e*E0d*n0d),
    "G_i_L": _bc_ends(gt["Gamma_i"])[1] / (u_e*E0d*n0d),
    "G_e_0": _bc_ends(gt["Gamma_e"])[0] / (u_e*E0d*n0d),
    "G_e_L": _bc_ends(gt["Gamma_e"])[1] / (u_e*E0d*n0d),
}
print("[BC]", BC_VALUES)

# labeled data candidates (optional) - only for n_i,n_e,V on nV grid
z_data_nV = jnp.asarray(z_test_nV, dtype=DTYPE)
n_i_data  = jnp.asarray(gt["n_i"]/n0d, dtype=DTYPE)
n_e_data  = jnp.asarray(gt["n_e"]/n0d, dtype=DTYPE)
V_data    = jnp.asarray(gt["V"]/(E0d*R0d), dtype=DTYPE)

z_minmax = jnp.array([0.0, float(Lz)], dtype=DTYPE)

[BC] {'n_i_0': 1.4190881805813176, 'n_i_L': 0.03607669102153166, 'n_e_0': 7.846056238979184e-06, 'n_e_L': 0.16701189226592, 'V_0': -18.095128179727826, 'V_L': 0.0, 'G_i_0': -0.005190036283461875, 'G_i_L': 0.0, 'G_e_0': 5.190036283461875e-05, 'G_e_L': 0.005241936631596912}


In [8]:
#%% ==========================================================
# 3) ICP sampler utilities (resampling + caching)
# ==========================================================
def _uniform(key, n, low, high):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    return low + (high - low) * u

def _left_heavy(key, n, low, high, power: float = 2.0):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    return low + (high - low) * (u ** power)

def _cheb_symmetric(key, n, low, high):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    x01 = 0.5 * (1.0 - jnp.cos(jnp.pi * u))
    return low + (high - low) * x01

def _sample_1d(key, n, low, high, mode: str = "uniform", power: float = 2.0):
    if mode == "uniform":
        return _uniform(key, n, low, high)
    elif mode == "left_heavy":
        return _left_heavy(key, n, low, high, power=power)
    elif mode == "cheb":
        return _cheb_symmetric(key, n, low, high)
    else:
        raise ValueError(f"Unknown mode: {mode}")

def sample_training_points_dc(
    key, N_r, N_lab,
    *, z_minmax,
    z_data_nV, n_i_data, n_e_data, V_data,
    use_chebyshev: bool=False,
    random_mode: str="uniform",
    z_power: float=2.0,
    z_r_fixed=None,
):
    mode_z_r = "cheb" if use_chebyshev else random_mode
    k1, k2 = jax.random.split(key, 2)

    z0, z1 = float(z_minmax[0]), float(z_minmax[1])

    if z_r_fixed is not None:
        z_r = z_r_fixed.astype(DTYPE)
    else:
        z_r = _sample_1d(k1, N_r, z0, z1, mode=mode_z_r, power=z_power)

    # interior data labels (only n_i,n_e,V)
    if N_lab and N_lab > 0:
        idx = jax.random.choice(k2, z_data_nV.shape[0], shape=(N_lab,), replace=False)
        z_lab   = z_data_nV[idx]
        ni_lab  = n_i_data[idx]
        ne_lab  = n_e_data[idx]
        V_lab   = V_data[idx]
    else:
        z_lab  = jnp.zeros((0,), dtype=DTYPE)
        ni_lab = jnp.zeros((0,), dtype=DTYPE)
        ne_lab = jnp.zeros((0,), dtype=DTYPE)
        V_lab  = jnp.zeros((0,), dtype=DTYPE)

    # Gamma data labels not used (kept for interface completeness)
    Gi_lab = jnp.zeros((0,), dtype=DTYPE)
    Ge_lab = jnp.zeros((0,), dtype=DTYPE)

    return (z_r, z_lab, ni_lab, ne_lab, V_lab, Gi_lab, Ge_lab)

In [9]:
#%% ==========================================================
# 4) PirateNet (ICP) with RWF + RFF + coord_norm
#    - IMPORTANT: only change is output dim 4->5
# ==========================================================
from flax.linen.initializers import glorot_normal, zeros
from typing import Callable, Dict, Union, Optional
from dataclasses import dataclass

def _weight_fact(init_fn: Callable, mean: float, stddev: float):
    def init(key, shape):
        key1, key2 = jax.random.split(key)
        w = init_fn(key1, shape)
        g = jax.nn.initializers.normal(stddev)(key2, (shape[-1],)) + mean
        g = jnp.exp(g).astype(w.dtype)
        v = (w / g).astype(w.dtype)
        return g, v
    return init

class Dense(nn.Module):
    features: int
    kernel_init: Callable = glorot_normal()
    bias_init: Callable = zeros
    reparam: Union[None, Dict] = None
    dtype: any = DTYPE
    param_dtype: any = DTYPE

    @nn.compact
    def __call__(self, x):
        in_dim = x.shape[-1]
        out_dim = self.features

        if self.reparam is None:
            kernel = self.param("kernel", self.kernel_init, (in_dim, out_dim)).astype(self.dtype)
        elif self.reparam["type"] == "weight_fact":
            g, v = self.param(
                "kernel",
                _weight_fact(self.kernel_init,
                             mean=self.reparam.get("mean", 1.0),
                             stddev=self.reparam.get("stddev", 0.1)),
                (in_dim, out_dim),
            )
            kernel = (g * v).astype(self.dtype)
        else:
            raise ValueError(f"Unknown reparam type: {self.reparam}")

        bias = self.param("bias", self.bias_init, (out_dim,)).astype(self.dtype)
        y = jnp.dot(x.astype(self.dtype), kernel) + bias
        return y

ACTS = {
    'tanh': jnn.tanh, 'relu': jnn.relu, 'gelu': jnn.gelu,
    'swish': jnn.swish, 'elu': jnn.elu, 'sigmoid': jnn.sigmoid, 'sin': jnp.sin
}

class PirateBlock(nn.Module):
    width: int
    act: str
    reparam: Union[None, Dict] = None

    @nn.compact
    def __call__(self, x, U, V):
        f = ACTS[self.act]
        h1 = f(Dense(self.width, reparam=self.reparam)(x))
        z1 = h1 * U + (1.0 - h1) * V
        h2 = f(Dense(self.width, reparam=self.reparam)(z1))
        z2 = h2 * U + (1.0 - h2) * V
        h  = f(Dense(self.width, reparam=self.reparam)(z2))
        alpha = self.param('alpha', nn.initializers.zeros, (), DTYPE)
        return alpha * h + (1.0 - alpha) * x

class PiratePINN_Network(nn.Module):
    width: int
    n_block: int
    act: str = 'tanh'
    fft: bool = False
    B: int = 64
    freq: float = 1.0
    coord_norm: bool = False
    z_minmax: Optional[jnp.ndarray] = None
    reparam: Union[None, Dict] = None

    def setup(self):
        in_dim = 1
        if self.fft:
            self.B_mat = self.param(
                'B_mat',
                jax.nn.initializers.normal(stddev=jnp.sqrt(2.0)),
                (self.B, in_dim), DTYPE
            )
        self.gate_U = Dense(self.width, reparam=self.reparam)
        self.gate_V = Dense(self.width, reparam=self.reparam)
        self.blocks = [PirateBlock(self.width, self.act, reparam=self.reparam)
                       for _ in range(self.n_block)]
        # >>> ONLY CHANGE: out dim = 5 <<<
        self.out = Dense(5, reparam=None)

    def _normalize(self, x):
        if not self.coord_norm:
            return x
        assert self.z_minmax is not None
        z0, z1 = self.z_minmax
        z = x[..., 0]
        z_ = (z - z0) / (z1 - z0 + 1e-12)
        return z_[..., None]

    def _fourier_embed(self, x):
        proj = 2*jnp.pi*self.freq*(x @ self.B_mat.T)
        return jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)

    def __call__(self, x):
        x = self._normalize(x) if self.coord_norm else x
        Phi = self._fourier_embed(x) if self.fft else x

        f = ACTS[self.act]
        U = f(self.gate_U(Phi))
        V = f(self.gate_V(Phi))

        h = Phi
        for blk in self.blocks:
            h = blk(h, U, V)

        y_raw = self.out(h)  # (...,5)
        return h, y_raw

@dataclass
class NetCfg:
    use_rwf: bool = True
    rwf_mean: float = 1.0
    rwf_std: float = 0.1

def reparam_from_cfg(cfg: NetCfg):
    return {"type": "weight_fact", "mean": cfg.rwf_mean, "stddev": cfg.rwf_std} if cfg.use_rwf else None

cfg = NetCfg(use_rwf=True)
model = PiratePINN_Network(
    width=WIDTH, n_block=DEPTH, act=ACT,
    fft=True, B=WIDTH//2, freq=1.0,
    coord_norm=True, z_minmax=z_minmax,
    reparam=reparam_from_cfg(cfg)
)

def _vars_for_apply(params):
    return params['nn'] if (isinstance(params, dict) and 'nn' in params) else params


In [10]:
#%% ==========================================================
# 5) DC hard BC apply (exp positivity + linear correction)
# ==========================================================
def model_apply_hard_bc_dc(params, z):
    variables = _vars_for_apply(params)

    z = jnp.asarray(z, dtype=DTYPE)
    z_shape = z.shape
    z_flat  = z.reshape(-1)
    inp     = z_flat[..., None]

    _, y_raw = model.apply(variables, inp)   # (N,5)
    free = jnp.exp(y_raw)                    # positivity

    z0 = jnp.zeros((1,1), dtype=DTYPE)
    zL = jnp.ones((1,1), dtype=DTYPE) * Lz
    _, y0_raw = model.apply(variables, z0)
    _, yL_raw = model.apply(variables, zL)
    free0 = jnp.exp(y0_raw[0])               # (5,)
    freeL = jnp.exp(yL_raw[0])               # (5,)

    bc0 = jnp.array([BC_VALUES["n_i_0"], BC_VALUES["n_e_0"], BC_VALUES["V_0"],
                     BC_VALUES["G_i_0"], BC_VALUES["G_e_0"]], dtype=DTYPE)
    bcL = jnp.array([BC_VALUES["n_i_L"], BC_VALUES["n_e_L"], BC_VALUES["V_L"],
                     BC_VALUES["G_i_L"], BC_VALUES["G_e_L"]], dtype=DTYPE)

    z_norm = (z_flat / (Lz + 1e-30))[:, None]
    y = free + (1.0 - z_norm) * (bc0 - free0) + z_norm * (bcL - freeL)
    y = y.reshape(*z_shape, 5)

    n_i, n_e, V, G_i, G_e = y[...,0], y[...,1], y[...,2], y[...,3], y[...,4]
    return n_i, n_e, V, G_i, G_e

model_apply = model_apply_hard_bc_dc

In [11]:
#%% ==========================================================
# 6) DC PDE residuals (autograd) - 5 terms
# ==========================================================
def mu_i(E):
    out = 0.5740/(1.0 + 0.66*jnp.sqrt(jnp.abs(E*E0d)*1e-2))
    out = out / u_e
    return out

def alpha_iz(E):
    out = 2922 * jnp.exp(-26.64*jnp.sqrt(1.0/(jnp.abs(E*E0d/100.) + 1e-30)))
    out = out / alpha0d
    return out

def residuals_dc(params, z):
    n_i, n_e, V, G_i, G_e = model_apply(params, z)

    dV_dz   = jax.grad(lambda zz: model_apply(params, zz)[2])(z)
    E       = -dV_dz
    d2V_dz2 = jax.grad(lambda zz: jax.grad(lambda zzz: model_apply(params, zzz)[2])(zz))(z)

    dn_i_dz = jax.grad(lambda zz: model_apply(params, zz)[0])(z)
    dn_e_dz = jax.grad(lambda zz: model_apply(params, zz)[1])(z)
    dG_i_dz = jax.grad(lambda zz: model_apply(params, zz)[3])(z)
    dG_e_dz = jax.grad(lambda zz: model_apply(params, zz)[4])(z)

    a  = alpha_iz(E)
    ui = mu_i(E)

    r1 = -dG_i_dz + a * G_e
    r2 = -dG_e_dz + a * G_e
    r3 = d2V_dz2 + (n_i - n_e)
    r4 = G_i - (ui*E*n_i - D_i*dn_i_dz)
    r5 = G_e - (-E*n_e - D_e*dn_e_dz)
    return r1, r2, r3, r4, r5

v_residuals = jax.vmap(residuals_dc, in_axes=(None, 0))

In [12]:
#%% ==========================================================
# 7) SAW (GradNorm) for 5 residuals (+ optional data)
# ==========================================================
@struct.dataclass
class SAWConfig:
    update_every: int = 1000
    ema_beta: float = 0.9
    eps: float = 1e-12
    include_data: bool = True
    normalize: bool = True

def _default_saw():
    return {
        'lambda_r1': jnp.array(1.0, DTYPE),
        'lambda_r2': jnp.array(1.0, DTYPE),
        'lambda_r3': jnp.array(1.0, DTYPE),
        'lambda_r4': jnp.array(1.0, DTYPE),
        'lambda_r5': jnp.array(1.0, DTYPE),
        'lambda_data': jnp.array(1.0, DTYPE),

        'g_r1_ema': jnp.array(1.0, DTYPE),
        'g_r2_ema': jnp.array(1.0, DTYPE),
        'g_r3_ema': jnp.array(1.0, DTYPE),
        'g_r4_ema': jnp.array(1.0, DTYPE),
        'g_r5_ema': jnp.array(1.0, DTYPE),
        'g_data_ema': jnp.array(1.0, DTYPE),
    }

class TrainState(flax_train_state.TrainState):
    saw: dict = struct.field(pytree_node=True,  default_factory=_default_saw)
    saw_cfg: SAWConfig = struct.field(pytree_node=False, default_factory=SAWConfig)

def _l2_norm_pytree(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum([jnp.sum(jnp.square(x)) for x in leaves]))

In [13]:
#%% ==========================================================
# 8) Loss computation (physics + optional data)
#    - data: only (n_i,n_e,V) on nV grid
# ==========================================================
def compute_losses_unweighted(params, batch):
    pnn = params['nn'] if (isinstance(params, dict) and 'nn' in params) else params
    (z_r, z_lab, ni_lab, ne_lab, V_lab, Gi_lab, Ge_lab) = batch

    r1, r2, r3, r4, r5 = v_residuals(pnn, z_r)
    L1 = jnp.mean(r1**2)
    L2 = jnp.mean(r2**2)
    L3 = jnp.mean(r3**2)
    L4 = jnp.mean(r4**2)
    L5 = jnp.mean(r5**2)
    L_phys = L1+L2+L3+L4+L5

    if z_lab.size == 0:
        Ld = jnp.array(0.0, DTYPE)
        Ld_ni = Ld_ne = Ld_V = jnp.array(0.0, DTYPE)
        Ld_Gi = Ld_Ge = jnp.array(0.0, DTYPE)
    else:
        ni_p, ne_p, V_p, Gi_p, Ge_p = jax.vmap(lambda zz: model_apply(pnn, zz))(z_lab)
        Ld_ni = jnp.mean((ni_p - ni_lab)**2)
        Ld_ne = jnp.mean((ne_p - ne_lab)**2)
        Ld_V  = jnp.mean((V_p  - V_lab )**2)
        # Gamma data loss disabled (per your request)
        Ld_Gi = jnp.array(0.0, DTYPE)
        Ld_Ge = jnp.array(0.0, DTYPE)
        Ld = Ld_ni + Ld_ne + Ld_V

    extras = dict(
        total=L_phys + Ld,
        phys_total=L_phys,
        phys_r1=L1, phys_r2=L2, phys_r3=L3, phys_r4=L4, phys_r5=L5,
        bc_total=jnp.array(0.0, DTYPE),
        data_total=Ld,
        data_ni=Ld_ni, data_ne=Ld_ne, data_V=Ld_V, data_Gi=Ld_Gi, data_Ge=Ld_Ge,
        phys_w1=1.0, phys_w2=1.0, phys_w3=1.0, phys_w4=1.0, phys_w5=1.0,
    )
    return L_phys, jnp.array(0.0, DTYPE), Ld, extras

def _grad_norm_of_term(params, batch, term: str):
    def term_loss(p):
        Lp, Lb, Ld, extras = compute_losses_unweighted(p, batch)
        if term == 'r1': return extras['phys_r1']
        if term == 'r2': return extras['phys_r2']
        if term == 'r3': return extras['phys_r3']
        if term == 'r4': return extras['phys_r4']
        if term == 'r5': return extras['phys_r5']
        if term == 'data': return Ld
        raise ValueError(term)
    g = jax.grad(term_loss)(params)
    return _l2_norm_pytree(g)

def _update_saw(state: TrainState, params, batch):
    cfg = state.saw_cfg
    beta = jnp.array(cfg.ema_beta, DTYPE)
    eps  = jnp.array(cfg.eps, DTYPE)
    s    = state.saw

    g1 = _grad_norm_of_term(params, batch, 'r1')
    g2 = _grad_norm_of_term(params, batch, 'r2')
    g3 = _grad_norm_of_term(params, batch, 'r3')
    g4 = _grad_norm_of_term(params, batch, 'r4')
    g5 = _grad_norm_of_term(params, batch, 'r5')
    gd = _grad_norm_of_term(params, batch, 'data') if cfg.include_data else jnp.array(0.0, DTYPE)

    g1e = beta*s['g_r1_ema'] + (1-beta)*g1
    g2e = beta*s['g_r2_ema'] + (1-beta)*g2
    g3e = beta*s['g_r3_ema'] + (1-beta)*g3
    g4e = beta*s['g_r4_ema'] + (1-beta)*g4
    g5e = beta*s['g_r5_ema'] + (1-beta)*g5
    gde = beta*s['g_data_ema'] + (1-beta)*gd if cfg.include_data else jnp.array(0.0, DTYPE)

    sum_g = g1e+g2e+g3e+g4e+g5e + (gde if cfg.include_data else 0.0) + eps

    lam1 = sum_g/(g1e+eps)
    lam2 = sum_g/(g2e+eps)
    lam3 = sum_g/(g3e+eps)
    lam4 = sum_g/(g4e+eps)
    lam5 = sum_g/(g5e+eps)
    lamd = sum_g/(gde+eps) if cfg.include_data else jnp.array(1.0, DTYPE)

    if cfg.normalize:
        n_terms = 5 + (1 if cfg.include_data else 0)
        denom = (lam1+lam2+lam3+lam4+lam5 + (lamd if cfg.include_data else 0.0)) / n_terms
        lam1,lam2,lam3,lam4,lam5 = lam1/denom,lam2/denom,lam3/denom,lam4/denom,lam5/denom
        lamd = lamd/denom if cfg.include_data else lamd

    new_saw = dict(
        lambda_r1=lam1, lambda_r2=lam2, lambda_r3=lam3, lambda_r4=lam4, lambda_r5=lam5,
        lambda_data=lamd,
        g_r1_ema=g1e, g_r2_ema=g2e, g_r3_ema=g3e, g_r4_ema=g4e, g_r5_ema=g5e, g_data_ema=gde,
    )
    return state.replace(saw=new_saw)

In [14]:
#%% ==========================================================
# 9) Optimizer/TrainState (SOAP + schedule + clip) - ICP style
# ==========================================================
def create_train_state(key, clip_norm=1.0, precondition_frequency=2, weight_decay=0.0):
    net_vars = model.init(key, jnp.ones((1,1), dtype=DTYPE))

    lr_or_schedule = optax.warmup_exponential_decay_schedule(
        init_value=jnp.array(0.0, dtype=DTYPE),
        peak_value=jnp.array(LR, dtype=DTYPE),
        warmup_steps=22_000,
        transition_steps=2_000,
        decay_rate=0.9,
        staircase=True,
    )

    tx = optax.chain(
        optax.clip_by_global_norm(clip_norm),
        soap(
            learning_rate=lr_or_schedule,
            b1=0.9, b2=0.999,
            weight_decay=weight_decay,
            precondition_frequency=precondition_frequency,
        )
    )

    return TrainState.create(
        apply_fn=None,
        params={'nn': net_vars},
        tx=tx,
        saw=_default_saw(),
        # N_lab=0이면 data loss 자체를 쓰지 않으므로, SAW도 data를 제외
        saw_cfg=SAWConfig(include_data=(N_lab > 0)),
    )

In [15]:
#%% ==========================================================
# 10) Hash / param save (ICP style)
# ==========================================================
def _hash_leaf(h, key, v):
    if isinstance(v, (jnp.ndarray, np.ndarray)):
        arr = np.asarray(v)
        h.update(str(arr.dtype).encode())
        h.update(np.asarray(arr.shape, dtype=np.int64).tobytes())
        h.update(arr.tobytes())
        return
    if isinstance(v, (tuple, list)):
        h.update(f"{key}#len={len(v)}".encode())
        for i, item in enumerate(v):
            _hash_leaf(h, f"{key}/{i}", item)
        return
    if isinstance(v, dict):
        for kk in sorted(v.keys()):
            _hash_leaf(h, f"{key}/{kk}", v[kk])
        return
    if v is None:
        h.update(b"None")
        return
    try:
        arr = np.asarray(v)
        h.update(str(arr.dtype).encode())
        h.update(np.asarray(arr.shape, dtype=np.int64).tobytes())
        h.update(arr.tobytes())
    except Exception:
        h.update(str(v).encode())

def hash_params(tree):
    flat = flatten_dict(tree, sep='/')
    h = hashlib.sha256()
    for k, v in sorted(flat.items()):
        h.update(k.encode())
        _hash_leaf(h, k, v)
    return h.hexdigest()

def save_params_npz(params, path: Path):
    flat = flatten_dict(params, sep='/')
    out = {}
    for k, v in flat.items():
        if isinstance(v, (jnp.ndarray, np.ndarray)):
            out[k] = np.asarray(v)
        elif isinstance(v, (tuple, list)):
            for i, item in enumerate(v):
                out[f"{k}__{i}"] = np.asarray(item)
        else:
            try:
                out[k] = np.asarray(v)
            except Exception:
                out[k] = np.asarray(str(v))
    np.savez(path, **out)

In [16]:
#%% ==========================================================
# 11) build_log_df (DC 5-field version)
# ==========================================================
def build_log_df(log):
    base_cols = [
        'step', 'total',
        'phys_total', 'phys_r1','phys_r2','phys_r3','phys_r4','phys_r5',
        'bc_total',
        'data_total','data_ni','data_ne','data_V','data_Gi','data_Ge',
        'lambda_r1','lambda_r2','lambda_r3','lambda_r4','lambda_r5','lambda_data',
    ]
    arr = []
    for row in log:
        arr.append([float(np.asarray(row[c])) for c in base_cols])
    df = pd.DataFrame(arr, columns=base_cols)
    return df

In [17]:
#%% ==========================================================
# 12) save_results_npz / overlay eval / plot bundle (DC version)
# ==========================================================
def _df_to_arrays(df: pd.DataFrame):
    out = {}
    for c in df.columns:
        out[f"log__{c}"] = np.asarray(df[c].to_numpy())
    out["log__columns"] = np.array(df.columns.to_list())
    return out

def _batch_to_arrays(batch):
    (z_r, z_lab, ni_lab, ne_lab, V_lab, Gi_lab, Ge_lab) = batch
    return dict(
        z_r=np.asarray(z_r),
        z_lab=np.asarray(z_lab),
        ni_lab=np.asarray(ni_lab),
        ne_lab=np.asarray(ne_lab),
        V_lab=np.asarray(V_lab),
    )

def _eval_overlay_arrays(state):
    pnn = state.params['nn']
    # predictions on nV grid
    z_nV = z_test_nV
    ni_p, ne_p, V_p, Gi_p, Ge_p = jax.vmap(lambda zz: model_apply(pnn, zz))(z_nV)
    # predictions on GE grid for Gamma
    z_GE = z_test_GE
    ni_p2, ne_p2, V_p2, Gi_p2, Ge_p2 = jax.vmap(lambda zz: model_apply(pnn, zz))(z_GE)

    # pack exact vs pred (SI + nondim both)
    out = dict(
        # grids
        z_nV_SI=np.asarray(z_test_nV_SI),
        z_GE_SI=np.asarray(z_test_GE_SI),
        z_nV=np.asarray(z_nV),
        z_GE=np.asarray(z_GE),

        # exact (SI)
        ni_exact=np.asarray(gt["n_i"]),
        ne_exact=np.asarray(gt["n_e"]),
        V_exact=np.asarray(gt["V"]),
        Gi_exact=np.asarray(gt["Gamma_i"]),
        Ge_exact=np.asarray(gt["Gamma_e"]),

        # pred (nondim)
        ni_pred_nV=np.asarray(ni_p),
        ne_pred_nV=np.asarray(ne_p),
        V_pred_nV=np.asarray(V_p),
        Gi_pred_GE=np.asarray(Gi_p2),
        Ge_pred_GE=np.asarray(Ge_p2),

        # pred (SI)
        ni_pred_SI=np.asarray(ni_p)*n0d,
        ne_pred_SI=np.asarray(ne_p)*n0d,
        V_pred_SI =np.asarray(V_p)*(E0d*R0d),
        Gi_pred_SI=np.asarray(Gi_p2)*(u_e*E0d*n0d),
        Ge_pred_SI=np.asarray(Ge_p2)*(u_e*E0d*n0d),
    )
    return out

def _eval_grid_prediction(state, ngrid=512):
    pnn = state.params['nn']
    z_grid = np.linspace(0.0, float(Lz), ngrid, dtype=float)
    ni_p, ne_p, V_p, Gi_p, Ge_p = jax.vmap(lambda zz: model_apply(pnn, zz))(jnp.asarray(z_grid, dtype=DTYPE))
    return dict(
        z_grid=np.asarray(z_grid),
        ni_grid=np.asarray(ni_p),
        ne_grid=np.asarray(ne_p),
        V_grid=np.asarray(V_p),
        Gi_grid=np.asarray(Gi_p),
        Ge_grid=np.asarray(Ge_p),
    )

def _find_param_B_mat(params_tree):
    # flatten and try to find any key ending with 'B_mat'
    flat = flatten_dict(params_tree, sep='/')
    for k, v in flat.items():
        if k.endswith("B_mat"):
            return np.asarray(v)
    return None

def save_results_npz(state, log, batch, step: int, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    df_log = build_log_df(log)

    pack = {
        "meta__step": np.array(step, dtype=np.int64),
        "meta__saved_at": np.array(_now()),
        "meta__hash": np.array(hash_params(state.params)),
        # DC constants
        "dc__n0d": np.array(n0d), "dc__E0d": np.array(E0d), "dc__R0d": np.array(R0d), "dc__t0d": np.array(t0d),
        "dc__u_e": np.array(u_e), "dc__D_i": np.array(D_i), "dc__D_e": np.array(D_e),
        "dc__alpha0d": np.array(alpha0d), "dc__Lz": np.array(float(Lz)), "dc__V_dc": np.array(V_dc),
        # BC
        "bc__n_i_0": np.array(BC_VALUES["n_i_0"]), "bc__n_i_L": np.array(BC_VALUES["n_i_L"]),
        "bc__n_e_0": np.array(BC_VALUES["n_e_0"]), "bc__n_e_L": np.array(BC_VALUES["n_e_L"]),
        "bc__V_0":   np.array(BC_VALUES["V_0"]),   "bc__V_L":   np.array(BC_VALUES["V_L"]),
        "bc__G_i_0": np.array(BC_VALUES["G_i_0"]), "bc__G_i_L": np.array(BC_VALUES["G_i_L"]),
        "bc__G_e_0": np.array(BC_VALUES["G_e_0"]), "bc__G_e_L": np.array(BC_VALUES["G_e_L"]),
    }

    B_mat = _find_param_B_mat(state.params)
    if B_mat is not None:
        pack["dc__B_mat"] = B_mat

    pack.update(_df_to_arrays(df_log))
    pack.update(_eval_overlay_arrays(state))
    pack.update(_eval_grid_prediction(state, ngrid=512))
    if batch is not None:
        pack.update(_batch_to_arrays(batch))

    npz_path = out_dir / f"results_{_tag(step)}.npz"
    np.savez(npz_path, **pack)
    print(f"[npz] saved → {npz_path}")
    return npz_path

# ---- plot helpers (DC version) ----
def _save_loss_plots(df_log: pd.DataFrame, out_dir: Path, step: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    def _plot(ycols, title, ylog, fname):
        plt.figure(figsize=(7,4))
        for c in ycols:
            if c in df_log.columns:
                plt.plot(df_log['step'], df_log[c], label=c)
        plt.xlabel("step"); plt.ylabel("value")
        plt.title(title)
        if ylog: plt.yscale("log")
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(out_dir/fname, dpi=160)
        plt.close()

    _plot(['total','phys_total','data_total'], "[Plot 1] Total & Grouped (DC)", True, f"loss_plot1_{_tag(step)}.png")
    _plot(['phys_r1','phys_r2','phys_r3','phys_r4','phys_r5'], "[Plot 2] Physics residuals (DC)", True, f"loss_plot2_{_tag(step)}.png")
    _plot(['lambda_r1','lambda_r2','lambda_r3','lambda_r4','lambda_r5','lambda_data'], "[Plot 3] SAW lambdas (DC)", True, f"loss_plot3_{_tag(step)}.png")
    _plot(['data_ni','data_ne','data_V','data_Gi','data_Ge'], "[Plot 4] Data breakdown (DC)", True, f"loss_plot4_{_tag(step)}.png")

def _save_overlay_plots(state, out_dir: Path, step: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    ov = _eval_overlay_arrays(state)

    # n_i, n_e, V on nV grid
    z_nV = ov["z_nV_SI"]
    def _ov1(x, exact, pred, title, ylab, fname):
        plt.figure(figsize=(7,4))
        plt.plot(x, exact, marker='o', ms=3, lw=0, label="Exact")
        plt.plot(x, pred, lw=1.8, label="PINN")
        plt.xlabel("z [m]"); plt.ylabel(ylab); plt.title(title)
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(out_dir/fname, dpi=160); plt.close()

    _ov1(z_nV, ov["ni_exact"], ov["ni_pred_SI"], "n_i (SI)", "n_i [m^-3]", f"overlay_ni_{_tag(step)}.png")
    _ov1(z_nV, ov["ne_exact"], ov["ne_pred_SI"], "n_e (SI)", "n_e [m^-3]", f"overlay_ne_{_tag(step)}.png")
    _ov1(z_nV, ov["V_exact"],  ov["V_pred_SI"],  "V (SI)",   "V [V]",      f"overlay_V_{_tag(step)}.png")

    # Gamma on GE grid
    z_GE = ov["z_GE_SI"]
    _ov1(z_GE, ov["Gi_exact"], ov["Gi_pred_SI"], "Gamma_i (SI)", "Gamma_i", f"overlay_Gi_{_tag(step)}.png")
    _ov1(z_GE, ov["Ge_exact"], ov["Ge_pred_SI"], "Gamma_e (SI)", "Gamma_e", f"overlay_Ge_{_tag(step)}.png")

def save_all_plots_silent(state, log, step: int, out_root: Path):
    plots_dir = out_root / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    df_log = build_log_df(log)
    _save_loss_plots(df_log, plots_dir, step)
    _save_overlay_plots(state, plots_dir, step)
    print(f"[plot] saved → {plots_dir}")

In [18]:
#%% ==========================================================
# 13) train_step (ICP style, 5 residuals)
# ==========================================================
@jax.jit
def train_step(state: TrainState, batch):
    def total_loss_fn(params):
        Lp, Lb, Ld, extras = compute_losses_unweighted(params, batch)
        lam = state.saw
        lam1 = lam['lambda_r1']; lam2 = lam['lambda_r2']; lam3 = lam['lambda_r3']
        lam4 = lam['lambda_r4']; lam5 = lam['lambda_r5']
        # physics part (always)
        phys_part = (lam1*extras['phys_r1'] + lam2*extras['phys_r2'] + lam3*extras['phys_r3'] +
                     lam4*extras['phys_r4'] + lam5*extras['phys_r5'])
        # data part (only if include_data=True)
        total = jax.lax.cond(
            state.saw_cfg.include_data,
            lambda _: phys_part + lam['lambda_data'] * Ld,
            lambda _: phys_part,
            operand=None
        )

        metrics = {
            'phys_total': Lp,
            'data_total': Ld,
            **extras,
            'lambda_r1': lam1, 'lambda_r2': lam2, 'lambda_r3': lam3, 'lambda_r4': lam4, 'lambda_r5': lam5,
            # 로그 호환: include_data=False이면 의미 없으니 1.0으로 기록(원하면 0.0도 가능)
            'lambda_data': jax.lax.cond(
                state.saw_cfg.include_data,
                lambda _: lam['lambda_data'],
                lambda _: jnp.array(1.0, DTYPE),
                operand=None
            ),
        }
        return total, metrics

    (loss, metrics), grads = jax.value_and_grad(total_loss_fn, has_aux=True)(state.params)
    new_state = state.apply_gradients(grads=grads)

    step_mod  = jnp.mod(new_state.step, new_state.saw_cfg.update_every)
    do_update = jnp.equal(step_mod, 0)
    new_state = jax.lax.cond(do_update, lambda s: _update_saw(s, s.params, batch), lambda s: s, new_state)

    return new_state, {'total': loss, **metrics}

In [19]:
#%% ==========================================================
# 14) Create state + resume
# ==========================================================
state = create_train_state(key)
try:
    state = checkpoints.restore_checkpoint(str(CKPT_DIR), target=state)
except Exception as e:
    print("[warn] restore failed. start fresh:", e)

print("[resume] step =", int(getattr(state, "step", 0)))
print("[resume] hash =", hash_params(state.params))

[resume] step = 0
[resume] hash = 8e752d83b4c48d46469104d1eb74da42973f70e7171b2eb086a7e2e1bdf27ac2


In [20]:
#%% ==========================================================
# 15) Training loop (resampling + caching + save best/npz/plots/ckpt)
# ==========================================================
log = []
best_loss = np.inf
start_step = int(getattr(state, "step", 0))
cached_z_r = None

for step in trange(start_step, EPOCHS, desc="training"):
    resample_r = (cached_z_r is None) or ((step % RESAMPLE_COLL_EVERY) == 0)
    z_r_fixed = None if resample_r else cached_z_r

    batch = sample_training_points_dc(
        jax.random.fold_in(key, step),
        N_r=N_r, N_lab=N_lab,
        z_minmax=z_minmax,
        z_data_nV=z_data_nV, n_i_data=n_i_data, n_e_data=n_e_data, V_data=V_data,
        use_chebyshev=False, random_mode="uniform", z_power=2.0,
        z_r_fixed=z_r_fixed
    )

    if resample_r:
        cached_z_r = batch[0]

    state, m = train_step(state, batch)

    if step % 200 == 0:
        phash = hash_params(state.params)
        record = {'step': step, 'params_hash': phash, **{k: float(v) for k, v in m.items()}}
        log.append(record)

        print(
            f"[{step:7d}] hash={phash[:10]} "
            f"total={record['total']:.3e} | phys={record['phys_total']:.3e} "
            f"(r1={record['phys_r1']:.2e}, r2={record['phys_r2']:.2e}, r3={record['phys_r3']:.2e}, r4={record['phys_r4']:.2e}, r5={record['phys_r5']:.2e}) "
            f"| data={record['data_total']:.3e}"
        )

        cur_loss = record['total']
        if (cur_loss < best_loss) and (step > 0):
            best_loss = cur_loss
            try:
                checkpoints.save_checkpoint(str(CKPT_DIR), target=state, step=step, overwrite=True, keep=3)
                print(f"[ckpt] saved best at step {step}")
            except Exception as e:
                print(f"[ckpt][warn] save failed: {e}")
                out = CKPT_DIR / f"state_best_step{int(step)}.flax"
                with open(out, "wb") as f:
                    f.write(serialization.to_bytes(state))
                print(f"[ckpt][fallback] saved FLAX → {out}")
            save_params_npz(_vars_for_apply(state.params), BEST_PATH)
            print(f"  ↳ [best] params saved → {BEST_PATH} | loss={best_loss:.3e}")

    # periodic npz
    if (step > 0) and (step % NPZ_SAVE_EVERY == 0):
        try:
            save_results_npz(state, log, batch, step, epoch_dir(step))
        except Exception as e:
            print(f"[npz][warn] failed at {step}: {e}")

    # plot epochs
    if step in PLOT_EPOCHS:
        try:
            save_all_plots_silent(state, log, step, epoch_dir(step))
            save_results_npz(state, log, batch, step, epoch_dir(step))
        except Exception as e:
            print(f"[plot/npz][warn] failed at {step}: {e}")

    # extra ckpt
    if (step > 0) and (step % EXTRA_CKPT_EVERY == 0):
        try:
            checkpoints.save_checkpoint(str(epoch_dir(step)), target=state, step=step, overwrite=True, keep=None)
            print(f"[ckpt-extra] saved at {step} → {epoch_dir(step)}")
        except Exception as e:
            out = CKPT_SAVED_DIR / f"state_extra_{_tag(step)}.flax"
            with open(out, "wb") as f:
                f.write(serialization.to_bytes(state))
            print(f"[ckpt-extra][fallback] saved FLAX → {out} (err={e})")

    # periodic ckpt
    if (step % 2000 == 0) and (step > max(start_step, 0)) and (step > 0):
        try:
            checkpoints.save_checkpoint(str(CKPT_DIR), target=state, step=step, overwrite=True, keep=5)
            print(f"[ckpt] periodic saved at step {step}")
        except Exception as e:
            out = CKPT_DIR / f"state_periodic_step{int(step)}.flax"
            with open(out, "wb") as f:
                f.write(serialization.to_bytes(state))
            print(f"[ckpt][fallback] saved FLAX → {out} (err={e})")

training:   0%|          | 0/300001 [00:00<?, ?it/s]

[      0] hash=8e752d83b4 total=2.291e+01 | phys=2.291e+01 (r1=1.69e+00, r2=2.19e-01, r3=1.22e+00, r4=1.80e+01, r5=1.79e+00) | data=0.000e+00


[    200] hash=9f9a0b5693 total=2.458e+01 | phys=2.458e+01 (r1=1.81e+00, r2=2.13e-01, r3=1.16e+00, r4=1.96e+01, r5=1.83e+00) | data=0.000e+00
[ckpt] saved best at step 200
  ↳ [best] params saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/ckpts_dc_pirate_saw/best_params.npz | loss=2.458e+01
[    400] hash=ca3a847125 total=1.917e+01 | phys=1.917e+01 (r1=1.38e+00, r2=2.04e-01, r3=1.11e+00, r4=1.49e+01, r5=1.53e+00) | data=0.000e+00
[ckpt] saved best at step 400
  ↳ [best] params saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/ckpts_dc_pirate_saw/best_params.npz | loss=1.917e+01
[    600] hash=2e3f75c447 total=1.358e+01 | phys=1.358e+01 (r1=1.08e+00, r2=1.14e-01, r3=5.46e-01, r4=1.09e+01, r5=9.89e-01) | data=0.000e+00
[ckpt] saved best at step 600
  ↳ [best] params saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/ckpts_dc_pirate_saw/best_params.npz | loss=1.358e+01
[    800] hash=5300d09c5c total=1.102e-01 | phys=1.102e-01 (r1=3.25e-02, r2=2.07e-02, r3=1.16e-02, r4=3.48e-02, r5=

/tmp/ipykernel_54231/3268681871.py:127: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  if ylog: plt.yscale("log")


[plot] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0010000/plots
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0010000/results_step0010000.npz
[ckpt] periodic saved at step 10000
[  10200] hash=997051454e total=2.509e-06 | phys=2.509e-06 (r1=3.30e-07, r2=5.94e-07, r3=6.67e-07, r4=4.46e-07, r5=4.72e-07) | data=0.000e+00
[  10400] hash=6e8bc25d5b total=1.809e-06 | phys=1.809e-06 (r1=2.38e-07, r2=9.88e-07, r3=2.67e-08, r4=2.92e-07, r5=2.64e-07) | data=0.000e+00
[  10600] hash=73a3bb3e43 total=1.676e-06 | phys=1.676e-06 (r1=1.88e-07, r2=9.39e-07, r3=8.39e-08, r4=2.55e-07, r5=2.10e-07) | data=0.000e+00
[  10800] hash=9bfa708fd5 total=1.320e-06 | phys=1.320e-06 (r1=2.26e-07, r2=4.74e-07, r3=5.42e-08, r4=2.85e-07, r5=2.81e-07) | data=0.000e+00
[  11000] hash=a5a869aea7 total=1.537e-06 | phys=1.537e-06 (r1=3.45e-07, r2=5.17e-07, r3=3.81e-08, r4=2.52e-07, r5=3.84e-07) | data=0.000e+00
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/C

...

[plot] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0200000/plots
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0200000/results_step0200000.npz
[ckpt] periodic saved at step 200000
[ 200200] hash=2c4206b0fb total=2.881e-07 | phys=2.881e-07 (r1=2.84e-07, r2=7.89e-10, r3=2.24e-11, r4=3.33e-09, r5=5.49e-12) | data=0.000e+00
[ 200400] hash=eb4432b4e9 total=3.296e-07 | phys=3.296e-07 (r1=3.21e-07, r2=8.53e-10, r3=3.46e-11, r4=7.71e-09, r5=8.68e-12) | data=0.000e+00
[ 200600] hash=8bc601e0de total=3.471e-07 | phys=3.471e-07 (r1=3.34e-07, r2=1.08e-09, r3=6.35e-11, r4=1.18e-08, r5=1.17e-11) | data=0.000e+00
[ 200800] hash=629410749d total=3.024e-07 | phys=3.024e-07 (r1=2.95e-07, r2=7.17e-10, r3=2.29e-11, r4=6.57e-09, r5=4.63e-12) | data=0.000e+00
[ 201000] hash=fcf307a32b total=4.216e-07 | phys=4.216e-07 (r1=4.06e-07, r2=8.25e-10, r3=1.20e-10, r4=1.42e-08, r5=6.36e-12) | data=0.000e+00
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/

/tmp/ipykernel_54231/3268681871.py:127: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  if ylog: plt.yscale("log")


[plot] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0250000/plots
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0250000/results_step0250000.npz
[ckpt] periodic saved at step 250000
[ 250200] hash=29179dabbc total=3.439e-07 | phys=3.439e-07 (r1=3.35e-07, r2=7.09e-10, r3=1.50e-11, r4=8.62e-09, r5=5.41e-12) | data=0.000e+00
[ 250400] hash=46381a81be total=3.644e-07 | phys=3.644e-07 (r1=3.50e-07, r2=1.01e-09, r3=8.79e-11, r4=1.29e-08, r5=1.16e-11) | data=0.000e+00
[ 250600] hash=3875b8eed8 total=3.986e-07 | phys=3.986e-07 (r1=3.89e-07, r2=7.36e-10, r3=2.21e-11, r4=8.66e-09, r5=4.50e-12) | data=0.000e+00
[ 250800] hash=1546a91245 total=4.596e-07 | phys=4.596e-07 (r1=4.45e-07, r2=9.96e-10, r3=1.85e-11, r4=1.38e-08, r5=1.11e-11) | data=0.000e+00
[ 251000] hash=9f92eca3c4 total=3.495e-07 | phys=3.495e-07 (r1=3.42e-07, r2=8.32e-10, r3=2.00e-11, r4=7.06e-09, r5=5.08e-12) | data=0.000e+00
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/

/tmp/ipykernel_54231/3268681871.py:127: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  if ylog: plt.yscale("log")


[plot] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0300000/plots
[npz] saved → /workspace/CODE/DC/steady/exp_dc_pirate_saw/CKPT_SAVED/step_0300000/results_step0300000.npz
[ckpt] periodic saved at step 300000


In [21]:
#%% ==========================================================
# 16) Post: log df + quick plots (optional)
# ==========================================================
if len(log) > 0:
    df_log = build_log_df(log)
    df_log.to_csv(exp/"training_log.csv", index=False)
    print("[done] log saved:", exp/"training_log.csv")

[done] log saved: /workspace/CODE/DC/steady/exp_dc_pirate_saw/training_log.csv
